# Standalone Qubit Spectroscopy — OPX1000 / MW-FEM

This notebook reproduces what the `03a_qubit_spectroscopy` Qualibrate node does,
but **without** the node/graph infrastructure.  It is self-contained and runs
directly as a Jupyter notebook.

**Flow**:
1. [Load QUAM state](#1-load-quam-state)
2. [Sweep parameters](#2-sweep-parameters)
3. [QUA program](#3-qua-program)
4. [Execute on hardware](#4-execute-on-hardware)
5. [Fetch data into xarray](#5-fetch-data-into-xarray)
6. [Analyse & fit](#6-analyse-and-fit)
7. [Plot](#7-plot)
8. [Optional: update QUAM state](#8-optional-update-quam-state)

---

### Note on the QUAM class

The `from quam_config import Quam` import used by the Qualibrate nodes refers to
the `quam_config/` folder **that lives next to the calibration scripts**
(i.e. `qua-libs/qualibration_graphs/superconducting/quam_config/`).  That folder
must be on `sys.path` (the Qualibrate project does this automatically via its
`pyproject.toml`).

**You are NOT forced to use it.**  You have two alternatives:

| Option | When to use | How |
|--------|-------------|-----|
| `quam_config.Quam` | You want the SRF-specific fields (`cavities`, `cavity_transmon_pairs`, `temp_calibration`) and the Octave loopback fix | `from quam_config import Quam` (requires the project `sys.path`) |
| `FixedFrequencyQuam` from `quam_builder` | Plain transmon + resonator — no SRF cavity extras needed | `from quam_builder.architecture.superconducting.qpu import FixedFrequencyQuam` |
| Local copy | You want to iterate on the class in your own folder | Copy `quam_config/my_quam.py` next to your notebook, rename, import locally |

All three options call `.load()` the same way and produce the same QUAM object
tree — only the extra SRF fields differ.  **This notebook uses the first option**
because the state.json was created with the SRF Quam, but you can swap it for
`FixedFrequencyQuam` if you only need the qubit and resonator.

## 0. Imports

In [ ]:
import sys
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from qm.qua import *
from qualang_tools.loops import from_array
from qualang_tools.results import fetching_tool, progress_counter
from qualang_tools.units import unit

u = unit(coerce_to_integer=True)

# --- QUAM class selection -------------------------------------------------------
# Option A (used here): local quam_config package in this folder.
# It extends FixedFrequencyQuam with SRF cavity fields.
_here = os.path.abspath('')
if _here not in sys.path:
    sys.path.insert(0, _here)
from quam_config import Quam

# Option B: plain FixedFrequencyQuam (no SRF extras, no local package needed)
# from quam_builder.architecture.superconducting.qpu import FixedFrequencyQuam as Quam

print("Quam class loaded:", Quam)

## 1. Load QUAM state

`Quam.load()` reads `quam_state/state.json` (and `wiring.json`) relative to the
path stored in the environment variable `QUAM_CONFIG_PATH`, or, if that is not
set, relative to the current working directory.  You can also pass a path
explicitly: `Quam.load('/path/to/state.json')`.

In [ ]:
machine = Quam.load()   # reads state/ folder in the project directory

# Inspect
qubit = machine.qubits["q1"]
print(f"Qubit f_01     : {qubit.f_01 / 1e9:.4f} GHz")
print(f"Qubit T1       : {qubit.T1 * 1e6:.1f} µs")
print(f"Resonator f_01 : {qubit.resonator.f_01 / 1e9:.4f} GHz")
print(f"Depletion time : {qubit.resonator.depletion_time} ns")
print(f"Saturation amp : {qubit.xy.operations['saturation'].amplitude:.4f} V")
print(f"Saturation len : {qubit.xy.operations['saturation'].length} ns")

## 2. Sweep parameters

Edit these to match your experiment.

In [ ]:
# Qubits to measure — use a subset of machine.qubits if needed
qubits = [machine.qubits["q1"]]
num_qubits = len(qubits)

# Averaging
n_avg = 100

# Frequency sweep (relative to each qubit's current IF)
span_MHz  = 100.0   # total span [MHz]
step_MHz  = 0.25    # step size [MHz]
dfs = np.arange(-span_MHz / 2 * u.MHz, span_MHz / 2 * u.MHz, step_MHz * u.MHz)

# Saturation pulse overrides (set to None to use values from the QUAM state)
operation            = "saturation"  # which qubit.xy operation to play
operation_amp_factor = 1.0           # amplitude pre-factor [-2, 2)
operation_len_ns     = None          # override pulse length [ns], or None

print(f"Sweep: {len(dfs)} points from {dfs[0]/1e6:.1f} to {dfs[-1]/1e6:.1f} MHz")
print(f"Estimated duration: ~{n_avg * len(dfs) * (qubit.xy.operations[operation].length + qubit.resonator.depletion_time) * 1e-9:.1f} s")

## 3. QUA program

Mirrors the `create_qua_program` action in `03a_qubit_spectroscopy.py`.
The QUAM channel methods (`qubit.xy.play`, `qubit.resonator.measure`, etc.) hide
all the low-level element names and intermediate-frequency bookkeeping.

In [ ]:
with program() as qua_prog:

    # ── Declare QUA variables ────────────────────────────────────────────────
    # machine.declare_qua_variables() returns lists indexed 0..num_qubits-1:
    #   I[i], I_st[i]  — I quadrature value and stream for qubit i
    #   Q[i], Q_st[i]  — Q quadrature value and stream for qubit i
    #   n, n_st        — averaging counter and its stream
    I, I_st, Q, Q_st, n, n_st = machine.declare_qua_variables()
    df = declare(int)  # QUA integer for the frequency detuning

    # ── Initialise QPU (set flux points for tunable elements, etc.) ──────────
    for qubit in qubits:
        machine.initialize_qpu(target=qubit)
    align()

    # ── Main sweep ───────────────────────────────────────────────────────────
    with for_(n, 0, n < n_avg, n + 1):
        save(n, n_st)
        with for_(*from_array(df, dfs)):

            # Drive each qubit at the swept frequency
            for i, qubit in enumerate(qubits):
                duration = (
                    operation_len_ns
                    if operation_len_ns is not None
                    else qubit.xy.operations[operation].length
                )
                # Shift qubit XY intermediate frequency by df
                qubit.xy.update_frequency(df + qubit.xy.intermediate_frequency)
                # Play the saturation pulse (duration in QUA clock cycles = ns/4)
                qubit.xy.play(
                    operation,
                    amplitude_scale=operation_amp_factor,
                    duration=duration // 4,
                )
            align()

            # Measure resonator and collect I/Q
            for i, qubit in enumerate(qubits):
                qubit.resonator.measure("readout", qua_vars=(I[i], Q[i]))
                # Wait for the resonator field to fully deplete before next shot
                qubit.resonator.wait(machine.depletion_time * u.ns)
                save(I[i], I_st[i])
                save(Q[i], Q_st[i])
            align()

    # ── Stream processing: buffer over frequency sweep, then average ─────────
    with stream_processing():
        n_st.save("n")
        for i in range(num_qubits):
            I_st[i].buffer(len(dfs)).average().save(f"I{i + 1}")
            Q_st[i].buffer(len(dfs)).average().save(f"Q{i + 1}")

print("QUA program compiled.")

## 4. Execute on hardware

`machine.connect()` opens a `QuantumMachinesManager` using the host IP and port
stored in `state.json`.  `machine.generate_config()` produces the full QUA
hardware config dict from the current QUAM state.

In [ ]:
qmm    = machine.connect()
config = machine.generate_config()

qm  = qmm.open_qm(config)
job = qm.execute(qua_prog)
print("Job started:", job.id)   # job.id is a property (str) in QM API v2, not a method

## 5. Fetch data into xarray

We use `fetching_tool` from `qualang_tools` (the low-level equivalent of what
`XarrayDataFetcher` wraps) and manually assemble the data into an
`xr.Dataset` that mirrors what the Qualibrate node would produce.

In [ ]:
# Build the list of result handles we want to read
result_keys = ["n"] + [f"I{i+1}" for i in range(num_qubits)] + [f"Q{i+1}" for i in range(num_qubits)]
results = fetching_tool(job, result_keys, mode="live")

# Live progress loop — updates while the job is running
while results.is_processing():
    fetched = results.fetch_all()
    n_current = fetched[0]  # averaging counter
    progress_counter(n_current, n_avg, start_time=results.get_start_time())

# Final fetch after job completion
fetched = results.fetch_all()
n_done = fetched[0]
I_raw  = [fetched[1 + i]             for i in range(num_qubits)]   # list of 1D arrays
Q_raw  = [fetched[1 + num_qubits + i] for i in range(num_qubits)]

print(f"Done: {n_done} averages, I shape = {I_raw[0].shape}")
print(job.execution_report())

In [ ]:
# ── Assemble xarray Dataset ──────────────────────────────────────────────────
# Dimensions: qubit × detuning  (same layout as the Qualibrate node)

qubit_names = [q.name for q in qubits]

# Stack all qubits along a new 'qubit' dimension
I_stack = np.stack(I_raw, axis=0)   # shape (num_qubits, n_points)
Q_stack = np.stack(Q_raw, axis=0)

coords = {
    "qubit"   : qubit_names,
    "detuning": dfs,
}

ds = xr.Dataset(
    {
        "I": (["qubit", "detuning"], I_stack),
        "Q": (["qubit", "detuning"], Q_stack),
    },
    coords=coords,
)
ds["detuning"].attrs = {"long_name": "detuning", "units": "Hz"}

# Compute IQ amplitude and phase
ds["IQ_abs"]   = np.sqrt(ds.I**2 + ds.Q**2)
ds["IQ_phase"] = np.arctan2(ds.Q, ds.I)

# Absolute RF frequency axis (for plotting)
full_freq = np.array([ds.detuning.values + q.xy.RF_frequency for q in qubits])
ds["full_freq"] = (["qubit", "detuning"], full_freq)
ds["full_freq"].attrs = {"long_name": "RF frequency", "units": "Hz"}

# PCA rotation: project I/Q onto the axis of maximum variance
# (this is what convert_IQ_to_V + add_amplitude_and_phase do in the node)
for q_name in qubit_names:
    I_q = ds.sel(qubit=q_name).I.values
    Q_q = ds.sel(qubit=q_name).Q.values
    # Find the angle that maximises variance in one quadrature
    idx_max = np.argmax(np.abs(ds.sel(qubit=q_name).IQ_abs.values - ds.sel(qubit=q_name).IQ_abs.mean().values))
    angle   = np.arctan2(Q_q[idx_max] - Q_q.mean(), I_q[idx_max] - I_q.mean())
    I_rot   = I_q * np.cos(angle) + Q_q * np.sin(angle)
    ds[f"I_rot_{q_name}"] = xr.DataArray(I_rot, dims="detuning", coords={"detuning": ds.detuning})
    ds[f"iw_angle_{q_name}"] = float(angle)

print(ds)

## 6. Analyse and fit

Fit a Lorentzian to the rotated I quadrature (`I_rot`) to extract the qubit
resonance frequency and FWHM.  Adjust `find_dip=True` for reflection readout
setups where the qubit appears as a dip.

In [ ]:
find_dip = False   # set True for SRF reflection readout (dip instead of peak)

def lorentzian(x, amplitude, center, hwhm, baseline):
    """Lorentzian peak:  amplitude / (1 + ((x - center)/hwhm)^2) + baseline"""
    return amplitude / (1.0 + ((x - center) / hwhm) ** 2) + baseline


fit_results = {}

for qubit in qubits:
    q_name  = qubit.name
    signal  = ds[f"I_rot_{q_name}"].values
    x       = ds.detuning.values

    if find_dip:
        signal = -signal   # invert so we always fit a peak

    # Initial parameter estimates
    baseline0   = np.median(signal)
    amplitude0  = signal.max() - baseline0
    center0     = x[np.argmax(signal)]
    hwhm0       = (x[-1] - x[0]) / 10

    try:
        popt, pcov = curve_fit(
            lorentzian, x, signal,
            p0=[amplitude0, center0, hwhm0, baseline0],
            maxfev=10000,
        )
        amplitude_fit, center_fit, hwhm_fit, baseline_fit = popt
        fwhm_fit    = 2 * abs(hwhm_fit)
        res_freq    = qubit.xy.RF_frequency + center_fit
        perr        = np.sqrt(np.diag(pcov))
        success     = (
            amplitude_fit > 0
            and abs(center_fit) < span_MHz / 2 * u.MHz
            and fwhm_fit < span_MHz * u.MHz
        )
        fit_results[q_name] = dict(
            frequency   = res_freq,
            center_df   = center_fit,
            fwhm        = fwhm_fit,
            amplitude   = amplitude_fit,
            iw_angle    = float(ds[f"iw_angle_{q_name}"]),
            success     = success,
            popt        = popt,
        )
        status = "OK" if success else "FAIL"
        print(f"{q_name}: [{status}]  f = {res_freq/1e9:.4f} GHz  "
              f"FWHM = {fwhm_fit/1e3:.0f} kHz  center_df = {center_fit/1e6:.2f} MHz")
    except RuntimeError as e:
        fit_results[q_name] = dict(success=False)
        print(f"{q_name}: FIT FAILED — {e}")

## 7. Plot

In [ ]:
%matplotlib widget

fig, axes = plt.subplots(num_qubits, 2, figsize=(12, 4 * num_qubits), squeeze=False)
fig.suptitle("Qubit spectroscopy", fontsize=14)

for row, qubit in enumerate(qubits):
    q_name  = qubit.name
    freq_GHz = ds.sel(qubit=q_name).full_freq.values / 1e9
    signal   = ds[f"I_rot_{q_name}"].values
    iq_abs   = ds.sel(qubit=q_name).IQ_abs.values

    # ── Left panel: rotated I quadrature with Lorentzian fit ──────────────
    ax = axes[row, 0]
    ax.plot(freq_GHz, signal if not find_dip else -signal, ".", ms=3, label="I_rot")
    if fit_results[q_name].get("success"):
        x_fine = np.linspace(ds.detuning.values[0], ds.detuning.values[-1], 500)
        y_fit  = lorentzian(x_fine, *fit_results[q_name]["popt"])
        if find_dip:
            y_fit = -y_fit
        freq_fine = (x_fine + qubit.xy.RF_frequency) / 1e9
        ax.plot(freq_fine, y_fit, "r-", lw=1.5, label="Lorentzian fit")
        ax.axvline(fit_results[q_name]["frequency"] / 1e9, color="red",
                   ls="--", lw=1, label=f"f = {fit_results[q_name]['frequency']/1e9:.4f} GHz")
    ax.set_xlabel("RF frequency (GHz)")
    ax.set_ylabel("I_rot (V)")
    ax.set_title(f"{q_name} — I_rot")
    ax.legend(fontsize=8)

    # ── Right panel: IQ amplitude ─────────────────────────────────────────
    ax = axes[row, 1]
    ax.plot(freq_GHz, iq_abs, ".", ms=3, color="C1")
    if fit_results[q_name].get("success"):
        ax.axvline(fit_results[q_name]["frequency"] / 1e9, color="red",
                   ls="--", lw=1)
    ax.set_xlabel("RF frequency (GHz)")
    ax.set_ylabel("|IQ| (V)")
    ax.set_title(f"{q_name} — |IQ|")

fig.tight_layout()
plt.show()

## 8. Optional: update QUAM state

If the fit is good, overwrite the qubit frequency in the QUAM state and save.
Comment this out if you just want to inspect the data without touching the state.

In [ ]:
UPDATE_STATE = False   # set True when you trust the fit result

if UPDATE_STATE:
    for qubit in qubits:
        q_name = qubit.name
        if not fit_results[q_name].get("success"):
            print(f"{q_name}: fit failed, skipping state update.")
            continue
        new_freq = fit_results[q_name]["frequency"]
        print(f"{q_name}: updating f_01  {qubit.f_01/1e9:.4f} GHz  →  {new_freq/1e9:.4f} GHz")
        qubit.f_01          = new_freq
        qubit.xy.RF_frequency = new_freq
        # Optionally update the integration weight angle
        new_angle = fit_results[q_name]["iw_angle"]
        qubit.resonator.operations["readout"].integration_weights_angle = new_angle

    machine.save()
    print("QUAM state saved.")
else:
    print("State update skipped (UPDATE_STATE=False).")